<a href="https://colab.research.google.com/github/pskarthikk/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — Content lifecycle

The paper reports that growing pages are younger on average than declining pages (185 days vs 228 days).

**Methodology question:** Where does the growing/declining label come from, and does the comparison account for other factors that may differ between newer and older pages? I would want to confirm that the label definition and comparison design support the observed age difference without implying that age itself causes growth or decline.

### Finding 2 — Reader engagement and rankings

The paper reports that reader engagement has a very small relationship with search position and does not support engagement as a direct ranking factor.

**Methodology question:** How much engagement data is missing, and could pages with available engagement data differ systematically from pages without it? I would want to understand the coverage of the engagement fields before generalizing the observed relationship to the full dataset.

**Overall audit stance:** These are methodology questions about how far the evidence can support the interpretation. They do not imply that the reported observations are incorrect.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [7]:
from pathlib import Path
import subprocess
import sys

repo_url = "https://github.com/pskarthikk/flyrank-ml-internship.git"
repo = Path("/content/flyrank-ml-internship")

# Clone the repository if this Colab runtime does not have it yet.
if not repo.exists():
    print("Repository not found. Cloning...")
    result = subprocess.run(
        ["git", "clone", repo_url, str(repo)],
        capture_output=True,
        text=True,
    )

    print(result.stdout)

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("Repository clone failed.")
else:
    print("Repository already exists:", repo)

# Make the repository scripts importable.
sys.path.insert(0, str(repo / "scripts"))

ml_utils = repo / "scripts/ml_utils.py"

print("\nml_utils.py exists:", ml_utils.exists())
print("Repository exists:", repo.exists())

assert ml_utils.exists(), "scripts/ml_utils.py was not found."

print("Repository setup: PASS")

Repository not found. Cloning...


ml_utils.py exists: True
Repository exists: True
Repository setup: PASS


In [12]:
from pathlib import Path

repo = Path("/content/flyrank-ml-internship")

print("Repository exists:", repo.exists())

if repo.exists():
    print("\nRepository contents:")
    print([p.name for p in repo.iterdir()][:20])

print("\nml_utils.py exists:",
      (repo / "scripts" / "ml_utils.py").exists())

print("Feature vector exists:",
      (repo / "data" / "processed" / "refresh_feature_vector.csv").exists())

Repository exists: True

Repository contents:
['submission', 'DATA_USE.md', 'notebooks', '.github', 'outputs', 'GUIDE.md', 'requirements.txt', 'CLAUDE.md', 'docs', 'scripts', 'README.md', 'skills', 'data', 'SETUP.md', 'AGENTS.md', '.gitignore', 'LICENSE', 'work', '.git']

ml_utils.py exists: True
Feature vector exists: False


In [14]:
from pathlib import Path
import numpy as np
import pandas as pd

repo = Path("/content/flyrank-ml-internship")

RAW_PATH = repo / "data/raw/content_refresh_anonymized.csv"
FEATURE_PATH = repo / "data/processed/refresh_feature_vector.csv"

# Exact model features from the Week-5 ml_utils.py
MODEL_NUMERIC_FEATURES = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

MODEL_CATEGORICAL_FEATURES = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier",
]

# Load raw Week-5 dataset
df = pd.read_csv(RAW_PATH)

# Recreate the Week-5 target
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

# Recreate the Week-5 log features
for source, target in [
    ("impressions_90d", "log_impressions_90d"),
    ("clicks_90d", "log_clicks_90d"),
    ("sessions_90d", "log_sessions_90d"),
    ("ai_sessions_90d", "log_ai_sessions_90d"),
]:
    values = pd.to_numeric(
        df[source],
        errors="coerce"
    ).fillna(0)

    df[target] = np.log1p(values)

# Check that every required feature exists
required_features = (
    MODEL_NUMERIC_FEATURES
    + MODEL_CATEGORICAL_FEATURES
)

missing_features = [
    col
    for col in required_features
    if col not in df.columns
]

print("Rows:", len(df))
print(
    "Declining rows:",
    int(df["is_declining_label"].sum())
)
print(
    "Declining rate:",
    round(df["is_declining_label"].mean(), 4)
)
print(
    "Missing model features:",
    missing_features
)

assert not missing_features

# Save the exact feature vector used by Week 5
keep_columns = [
    "content_id",
    "client_id",
    "is_declining_label",
] + required_features

feature_df = df[keep_columns].copy()

FEATURE_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

feature_df.to_csv(
    FEATURE_PATH,
    index=False
)

print("\nFeature vector written successfully.")
print("Path:", FEATURE_PATH)
print("Shape:", feature_df.shape)
print("\nFeature preparation: PASS")

Rows: 30000
Declining rows: 16262
Declining rate: 0.5421
Missing model features: []

Feature vector written successfully.
Path: /content/flyrank-ml-internship/data/processed/refresh_feature_vector.csv
Shape: (30000, 29)

Feature preparation: PASS


In [15]:
# Section 2 — Before/after validation comparison

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier


# ---------------------------------------------------------
# Evaluation function
# ---------------------------------------------------------

def precision_at_k_local(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    order = np.argsort(-scores, kind="mergesort")
    top_k = order[:k]

    return float(y_true[top_k].mean())


def evaluate_ranking(y_true, scores):
    return {
        "Precision@20": precision_at_k_local(
            y_true, scores, 20
        ),
        "Precision@50": precision_at_k_local(
            y_true, scores, 50
        ),
        "Precision@100": precision_at_k_local(
            y_true, scores, 100
        ),
    }


# ---------------------------------------------------------
# Build model matrix
# ---------------------------------------------------------

numeric_frame = (
    feature_df[MODEL_NUMERIC_FEATURES]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

categorical_frame = (
    feature_df[MODEL_CATEGORICAL_FEATURES]
    .fillna("unknown")
    .astype(str)
)

encoded_categorical = pd.get_dummies(
    categorical_frame,
    prefix=MODEL_CATEGORICAL_FEATURES,
    dummy_na=False,
    dtype=float,
)

X = pd.concat(
    [
        numeric_frame.reset_index(drop=True),
        encoded_categorical.reset_index(drop=True),
    ],
    axis=1,
)

y = feature_df["is_declining_label"].astype(int)

print("Model matrix:", X.shape)


# ---------------------------------------------------------
# BEFORE — row-level random split
# ---------------------------------------------------------

rng = np.random.default_rng(42)

all_indices = np.arange(len(feature_df))
random_indices = rng.permutation(all_indices)

random_split = int(len(random_indices) * 0.80)

random_train = random_indices[:random_split]
random_test = random_indices[random_split:]

random_model = RandomForestClassifier(
    class_weight="balanced_subsample",
    max_depth=10,
    min_samples_leaf=25,
    n_estimators=200,
    n_jobs=-1,
    random_state=42,
)

random_model.fit(
    X.iloc[random_train],
    y.iloc[random_train],
)

random_scores = random_model.predict_proba(
    X.iloc[random_test]
)[:, 1]

before = evaluate_ranking(
    y.iloc[random_test],
    random_scores,
)


# ---------------------------------------------------------
# AFTER — client-grouped split
# ---------------------------------------------------------

client_series = (
    feature_df["client_id"]
    .fillna("unknown")
    .astype(str)
)

unique_clients = (
    client_series
    .drop_duplicates()
    .to_numpy()
)

# Reproduce Week-5 client holdout exactly.
rng = np.random.default_rng(42)

shuffled_clients = rng.permutation(
    unique_clients
)

test_client_count = max(
    1,
    int(round(len(shuffled_clients) * 0.20))
)

test_clients = set(
    shuffled_clients[:test_client_count]
)

group_test_mask = client_series.isin(
    test_clients
)

group_train = np.flatnonzero(
    ~group_test_mask.to_numpy()
)

group_test = np.flatnonzero(
    group_test_mask.to_numpy()
)

group_model = RandomForestClassifier(
    class_weight="balanced_subsample",
    max_depth=10,
    min_samples_leaf=25,
    n_estimators=200,
    n_jobs=-1,
    random_state=42,
)

group_model.fit(
    X.iloc[group_train],
    y.iloc[group_train],
)

group_scores = group_model.predict_proba(
    X.iloc[group_test]
)[:, 1]

after = evaluate_ranking(
    y.iloc[group_test],
    group_scores,
)


# ---------------------------------------------------------
# Verify there is no client leakage across the split
# ---------------------------------------------------------

train_clients = set(
    client_series.iloc[group_train]
)

test_clients = set(
    client_series.iloc[group_test]
)

overlap = train_clients.intersection(
    test_clients
)

assert len(overlap) == 0


# ---------------------------------------------------------
# Before / after table
# ---------------------------------------------------------

comparison = pd.DataFrame(
    [
        {
            "Validation": "Before — row-level random",
            **before,
        },
        {
            "Validation": "After — client-grouped",
            **after,
        },
    ]
)

print("\n" + "=" * 80)
print("BEFORE / AFTER VALIDATION")
print("=" * 80)

print(
    comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)

print("\nBefore test rows:", len(random_test))
print("After test rows:", len(group_test))
print("Held-out clients:", len(test_clients))
print("Client overlap:", len(overlap))

print("\nClient-grouped validation: PASS")

Model matrix: (30000, 52)

BEFORE / AFTER VALIDATION
               Validation  Precision@20  Precision@50  Precision@100
Before — row-level random         0.950         0.960          0.970
   After — client-grouped         0.650         0.740          0.720

Before test rows: 6000
After test rows: 2325
Held-out clients: 6
Client overlap: 0

Client-grouped validation: PASS


### Validation interpretation

The row-level random split measured Precision@20 of 0.950, Precision@50 of 0.960, and Precision@100 of 0.970. The client-grouped split measured 0.650, 0.740, and 0.720 at the same cutoffs.

The difference is directional evidence that row-level validation can give a more optimistic estimate when pages from the same client are present in both training and test data. The grouped split held out six clients and had zero client overlap between training and test sets.

For this audit, I treat the client-grouped results as the more appropriate measured performance for decision-support on unseen clients. These results are observed validation measurements, not a guarantee of future performance.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [16]:
# Section 3 — Leakage audit

print("=" * 80)
print("LEAKAGE AUDIT")
print("=" * 80)

# ---------------------------------------------------------
# 1. Final feature set used by the model
# ---------------------------------------------------------

final_features = (
    MODEL_NUMERIC_FEATURES
    + MODEL_CATEGORICAL_FEATURES
)

print("Number of final model features:", len(final_features))

print("\nFinal model features:")
for feature in final_features:
    print(" -", feature)


# ---------------------------------------------------------
# 2. Direct label / outcome fields that must not be features
# ---------------------------------------------------------

known_leakage_fields = [
    "is_declining_label",
    "trend_direction",
    "trend_pct",
]

leakage_in_features = [
    field
    for field in known_leakage_fields
    if field in final_features
]

print("\nKnown outcome-related fields found in model features:")
print(leakage_in_features)

assert len(leakage_in_features) == 0


# ---------------------------------------------------------
# 3. Check that the target itself is excluded
# ---------------------------------------------------------

assert "is_declining_label" not in final_features


# ---------------------------------------------------------
# 4. Check that trend fields are excluded
# ---------------------------------------------------------

assert "trend_direction" not in final_features
assert "trend_pct" not in final_features


# ---------------------------------------------------------
# 5. Look for suspicious feature names
# ---------------------------------------------------------

suspicious_terms = [
    "label",
    "target",
    "trend",
    "declin",
    "future",
    "outcome",
]

suspicious_features = [
    feature
    for feature in final_features
    if any(
        term in feature.lower()
        for term in suspicious_terms
    )
]

print("\nFeatures with potentially suspicious names:")
print(suspicious_features)


# ---------------------------------------------------------
# 6. Final audit result
# ---------------------------------------------------------

print("\n" + "=" * 80)
print("LEAKAGE AUDIT RESULT")
print("=" * 80)

print("Direct target leakage: NONE")
print("Trend/outcome fields in final features: NONE")

if suspicious_features:
    print(
        "\nSuspicious names require human review:",
        suspicious_features,
    )
else:
    print(
        "\nNo suspicious feature names detected."
    )

print(
    "\nConclusion: The final feature list does not contain "
    "the target or the fields directly used to construct the target."
)

print(
    "This is a feature-name audit; it does not prove that "
    "all possible forms of leakage are absent."
)

LEAKAGE AUDIT
Number of final model features: 26

Final model features:
 - search_volume
 - competition
 - cpc
 - word_count
 - char_count
 - log_impressions_90d
 - log_clicks_90d
 - log_sessions_90d
 - log_ai_sessions_90d
 - days_with_impressions
 - days_with_sessions
 - content_age_days
 - days_since_last_update
 - ctr
 - avg_position
 - engagement_rate
 - scroll_rate
 - ai_traffic_pct
 - competition_level
 - content_type
 - main_intent
 - age_tier
 - freshness_tier
 - word_count_tier
 - impression_tier
 - position_tier

Known outcome-related fields found in model features:
[]

Features with potentially suspicious names:
[]

LEAKAGE AUDIT RESULT
Direct target leakage: NONE
Trend/outcome fields in final features: NONE

No suspicious feature names detected.

Conclusion: The final feature list does not contain the target or the fields directly used to construct the target.
This is a feature-name audit; it does not prove that all possible forms of leakage are absent.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.